In [0]:
%python
import requests
from pyspark.sql.functions import current_timestamp

api_key = "e9e16e07083875eb883ce4f6b526a17d"

url = "https://api.themoviedb.org/3/movie/popular"

all_movies = []

start_page = 101
end_page = 151

for page in range(start_page, end_page):
    params = {
        "api_key": api_key,
        "language": "en-US",
        "page": page
    }

    response = requests.get(url, params=params)
    data = response.json()

    for movie in data["results"]:
        movie["source_page"] = page

    all_movies.extend(data["results"])

df = spark.createDataFrame(all_movies)

df_full = df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

# guardado en volumen (DATA LAKE)
df_full.write \
    .mode("append") \
    .json("/Volumes/analisis_tmdb/bronze/archivos/movies/")